In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
!pip install biopython
from Bio import SeqIO
from Bio.Seq import Seq
import os
from scipy.stats import mannwhitneyu
from Bio.Data import CodonTable
import itertools
import tqdm

In [4]:
split_positions = split_positions = {
    "SUP35": "1-135",
    "CYC8": "443-682",
    "RNQ1": "153-405",
    "URE2": '1-89',
    "NEW1": '1-153',
    'SWI1': '1-323',
    'MOD5': '194-215',
    'MOT3': '1-295',
    'GLN3': '166-242',
    'SFP1': '230-458',
    'SKY1': '353-491',
    'PUB1': '290-386',
    'PIN2': '221-282',
    'AZF1': '1-300',
    'SNT1': '1-1226',
    'RBS1': '260-400',
    'PSP1': '50-400'
}

# Parsing the fasta file and converting it into a dataframe that is easy to work with.

In [11]:
fasta_path = 'orf_coding_all_R64-4-1_20230830.fasta'

data = []

fasta_file = open(fasta_path, 'r')
for record in SeqIO.parse(fasta_file, 'fasta'):
    record_id = record.id
    prion_or_non_prion_info = record.description
    gene_info = record.description.split(',')
    sequence = str(record.seq)
    gene_name = gene_info[0].split(' ')[1]
    orf_types = ['Dubious ORF', 'Uncharacterized ORF', 'Verified ORF']
    orf_type = None
    for item in gene_info:
      match = re.search(rf'\b({"|".join(orf_types)})\b', item, re.IGNORECASE)
      if match:
        orf_type = match.group(0)
        break

    data.append([record_id, gene_name, orf_type, prion_or_non_prion_info, sequence])

df = pd.DataFrame(data, columns=['ID', 'Gene Name', 'ORF', 'Prion Formation', 'Sequence'])
df = df[df['ORF'] == 'Verified ORF'] # Filter anything that's not a verified ORF
# print(df['ORF'].value_counts()) # we have 5254 verified ORFs, 684 Dubious ORFs, and 669 Uncharacterized ORFs
df['Prion Formation'] = df['Prion Formation'].apply(lambda x: 1 if "+]" in x else 0)
prion_genes_not_caught_by_plus_symbol = ['SUP35', 'GLN3', 'YLR278C', 'SKY1', 'URE2', 'PUB1', 'MOD5']
false_prion_genes_caught_by_plus_symbol = ['PIN3', 'PMA1', 'HSP104', 'VTS1']
df.loc[df['Gene Name'].isin(prion_genes_not_caught_by_plus_symbol), 'Prion Formation'] = 1
df.loc[df['Gene Name'].isin(false_prion_genes_caught_by_plus_symbol), 'Prion Formation'] = 0

def validate_sequence(seq):
    divisible_by_3 = len(seq) % 3 == 0

    starts_with_atg = seq.startswith('ATG')

    ends_with_valid_codon = seq.endswith(('TAA', 'TAG', 'TGA'))

    return divisible_by_3 and starts_with_atg and ends_with_valid_codon

all_valid = df['Sequence'].apply(validate_sequence).all()

if all_valid:
    print("All sequences are valid!")
else:
    print("Some sequences are invalid.")
# We ensure that all sequences are valid!

All sequences are valid!


In [12]:
df.head()

,ID,Gene Name,ORF,Prion Formation,Sequence
2,YAL068C,PAU8,Verified ORF,0,ATGGTCAAATTAACTTCAATCGCCGCTGGTGTCGCTGCCATCGCTG...
4,YAL067C,SEO1,Verified ORF,0,ATGTATTCAATTGTTAAAGAGATTATTGTAGATCCTTACAAAAGAC...
9,YAL064W,YAL064W,Verified ORF,0,ATGAATCCTTTTGCATCGCTAGAAGGACAAGATAATATTTCTTCTG...
11,YAL063C,FLO9,Verified ORF,0,ATGTCTCTGGCACATTATTGTTTACTACTAGCCATCGTCACATTGC...
12,YAL062W,GDH3,Verified ORF,0,ATGACAAGCGAACCAGAGTTTCAGCAGGCTTACGATGAGATCGTTT...


In [13]:
print(len(df) - sum(df['Prion Formation'])) # we have 5237 non-prion-forming genes
print(sum(df['Prion Formation'])) # we have 17 prion-forming genes and

5237
17


# Calculating DNA Composition (each nucleotide percentage)

In [14]:
nucleotides = ['A', 'T', 'G', 'C']

def count_nucleotides(dna_sequence):
    return {nuc: (dna_sequence.count(nuc) / len(dna_sequence)) * 100 for nuc in nucleotides}

def calculate_nucleotide_composition(df):
 nucleotide_results = []

 for _, row in df.iterrows():
    counts = count_nucleotides(row['Sequence'])
    counts['ID'] = row['ID']  # the ID is needed for later merging
    nucleotide_results.append(counts)

 nucleotide_counts_df = pd.DataFrame(nucleotide_results)

 nucleotide_counts_df.rename(columns={'A': 'Ad', 'T': 'Th', 'G': 'Gu', 'C': 'Cy'}, inplace=True)

 return nucleotide_counts_df

# for entire dataset
nucleotide_counts_df = calculate_nucleotide_composition(df)
df = pd.merge(df, nucleotide_counts_df, on='ID')

# Translating the DNA sequences of the genes

In [15]:
def translate_dna(seq):
    return str(Seq(seq).translate(to_stop=True))
df['Protein Sequence'] = df['Sequence'].apply(translate_dna)

# Calculating the protein composition for every gene

In [16]:
amino_acids = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']

def count_amino_acids(protein_sequence):
    # Calculate percentage for each amino acid
    return {aa: (protein_sequence.count(aa) / len(protein_sequence)) * 100 for aa in amino_acids}

def calculate_amino_acid_composition(df):
 amino_acid_results = []

 for _, row in df.iterrows():
    counts = count_amino_acids(row['Protein Sequence'])
    counts['ID'] = row['ID']  # the ID is needed for later merging
    amino_acid_results.append(counts)

 amino_acid_counts_df = pd.DataFrame(amino_acid_results)
 return amino_acid_counts_df

# for entire dataset
amino_acid_counts_df = calculate_amino_acid_composition(df)
amino_acid_df = pd.merge(df, amino_acid_counts_df, on='ID')
amino_acid_df['Sequence'] = amino_acid_df['Sequence'].astype(str)

# A random sampling function.

In [17]:
def random_sample(df, N):

    sampled_df = df.sample(n=N, random_state=None)

    return sampled_df

In [18]:
random_df = random_sample(amino_acid_df[amino_acid_df['Prion Formation'] == 0], 17)
random_df

,ID,Gene Name,ORF,Prion Formation,Sequence,Ad,Th,Gu,Cy,Protein Sequence,...,L,K,M,F,P,S,T,W,Y,V
317,YBR162W-A,YSY6,Verified ORF,0,ATGGCCGTACAGACACCAAGACAAAGACTGGCTAATGCCAAGTTTA...,34.848485,24.747475,22.727273,17.676768,MAVQTPRQRLANAKFNKNNEKYRKYGKKKEGKTEKTAPVISKTWLG...,...,13.846154,15.384615,1.538462,3.076923,3.076923,3.076923,6.153846,1.538462,4.615385,6.153846
3210,YLR025W,SNF7,Verified ORF,0,ATGTGGTCATCACTTTTTGGTTGGACATCAAGTAATGCCAAGAATA...,38.727524,21.161826,24.619640,15.491010,MWSSLFGWTSSNAKNKESPTKAIVRLREHINLLSKKQSHLRTQITN...,...,9.166667,9.166667,4.166667,1.250000,2.083333,8.750000,5.000000,0.833333,0.000000,4.583333
1579,YFR016C,AIP5,Verified ORF,0,ATGGTAGAATCTTTGACTGTAGAAAATCAGGAACACAATGTTCAAC...,40.410589,20.043220,22.906537,16.639654,MVESLTVENQEHNVQPPSVTSAGDSYSTLATDLPLPSTNDIIESRD...,...,4.785077,9.651257,0.892133,1.622060,3.730738,7.866991,7.542579,0.162206,0.729927,6.244931
2652,YJL057C,IKS1,Verified ORF,0,ATGAGTTTAGTACCATATGAAGAAGGTTCGCTGATTTTAGATGATC...,33.133733,28.692615,20.209581,17.964072,MSLVPYEEGSLILDDPNSKSVVVVNPTSGTLSFFQQDNSNNDSELN...,...,9.745127,5.997001,1.799100,5.547226,4.347826,10.494753,5.247376,0.599700,3.148426,5.247376
2975,YKL062W,MSN4,Verified ORF,0,ATGCTAGTCTTCGGACCTAATAGTAGTTTCGTTCGTCACGCAAACA...,33.914422,26.677232,17.115689,22.292657,MLVFGPNSSFVRHANKKQEDSSIMNEPNGLMDPVLSTTNVSATSSN...,...,6.349206,6.190476,2.539683,4.444444,5.714286,14.603175,7.777778,0.476190,1.428571,3.015873
710,YDL100C,GET3,Verified ORF,0,ATGGATTTAACCGTGGAACCTAATTTGCACTCTTTAATTACCTCTA...,32.018779,28.262911,20.469484,19.248826,MDLTVEPNLHSLITSTTHKWIFVGGKGGVGKTTSSCSIAIQMALSQ...,...,11.016949,6.779661,3.672316,4.802260,3.107345,6.497175,6.497175,0.564972,1.694915,4.519774
2248,YHR121W,LSM12,Verified ORF,0,ATGAGTGTCAGCCTTGAGCAAACGCTCGGATTCAGAATAAAAGTTA...,37.056738,24.113475,21.985816,16.843972,MSVSLEQTLGFRIKVTNVLDVVTEGRLYSFNSSNNTLTIQTTKKNQ...,...,7.486631,12.299465,0.534759,4.812834,2.673797,8.021390,5.882353,1.069519,1.604278,9.625668
3764,YMR075W,RCO1,Verified ORF,0,ATGGACACCAGCAAGAAAGATACTACTAGGTCGCCCTCACATTCCA...,38.637470,27.104623,15.961071,18.296837,MDTSKKDTTRSPSHSNSSSPSSSSLSSSSSKEKKRPKRLSSQNVNY...,...,7.748538,11.257310,1.023392,4.532164,4.532164,12.134503,5.116959,1.169591,2.777778,2.777778
2523,YJL214W,HXT8,Verified ORF,0,ATGACTGATCGTAAAACCAACTTGCCAGAAGAACCGATTTTCGAAG...,24.327485,34.795322,23.274854,17.602339,MTDRKTNLPEEPIFEEAEDDGCPSIENSSHLSVPTVEENKDFSEYN...,...,7.381371,4.393673,3.690685,7.557118,3.690685,8.260105,6.326889,1.933216,4.745167,8.260105
2621,YJL092W,SRS2,Verified ORF,0,ATGTCGTCGAACAATGATCTTTGGTTGCATTTAGTATCCCAGTTAA...,35.432624,26.638298,19.744681,18.184397,MSSNNDLWLHLVSQLNTQQRAAALFDYTRGLQVIAGPGTGKTKVLT...,...,10.477002,9.199319,1.788756,4.173765,4.173765,9.028961,5.281090,0.511073,2.810903,3.918228


# Adding amino acids with their correspodning codons so that we understand codon bias

In [19]:
# Standard codon table
codon_table = CodonTable.unambiguous_dna_by_id[1]

# Mapping of codons to amino acids (excluding stop codons)
codon_to_aa = codon_table.forward_table

all_codons = {f"{aa}_{codon}" for codon, aa in codon_to_aa.items()}

def codon_count_per_aa(seq):
    codons = [seq[i:i+3] for i in range(0, len(seq), 3)]
    if seq.endswith(('TAA', 'TAG', 'TGA')):
        num_codons = len(codons) - 1
    else:
        num_codons = len(codons)

    codon_counts = {col: 0 for col in all_codons}

    for codon in codons:
        if codon in codon_to_aa:
            aa = codon_to_aa[codon]
            column_name = f"{aa}_{codon}"
            codon_counts[column_name] += 1

    # Normalize by the total number of codons
    return {key: ((value / num_codons) * 100) if num_codons > 0 else 0 for key, value in codon_counts.items()}

def add_codon_bias(df):
    codon_results = []

    for _, row in df.iterrows():
        counts = codon_count_per_aa(row['Sequence'])
        counts['ID'] = row['ID']  # Keep ID for merging
        codon_results.append(counts)

    codon_df = pd.DataFrame(codon_results).fillna(0)
    return codon_df

codon_df = add_codon_bias(amino_acid_df)
complete_df = pd.merge(amino_acid_df, codon_df, on='ID')

# GC Content to be compared between the two gene groups.

In [20]:
complete_df['GC_Content'] = complete_df['Gu'] + complete_df['Cy']

# C and G in 1st or 2nd position codon frequency comparison.

In [21]:
def contains_in_first_or_second_pos(col_name, letter):
    parts = col_name.split('_') # because this is how I named my columns
    suffix = parts[1]
    return letter in suffix[:2] # Check the first and second characters of the suffix

# complete_df.iloc[:, 30:-1] --> This was used to slice only the columns with the codons
columns_with_c_at_first_or_second_pos = [col for col in complete_df.iloc[:, 30:-1].columns if contains_in_first_or_second_pos(col, 'C')]
columns_without_c_at_first_or_second_pos = [col for col in complete_df.iloc[:, 30:-1].columns if not contains_in_first_or_second_pos(col, 'C')]

complete_df['Codons_with_c_at_first_or_second_pos'] = complete_df[columns_with_c_at_first_or_second_pos].sum(axis=1)
complete_df['Codons_without_c_at_first_or_second_pos'] = complete_df[columns_without_c_at_first_or_second_pos].sum(axis=1)

In [22]:
columns_with_g_at_first_or_second_pos = [col for col in complete_df.iloc[:, 30:-3].columns if contains_in_first_or_second_pos(col, 'G')]
columns_without_g_at_first_or_second_pos = [col for col in complete_df.iloc[:, 30:-3].columns if not contains_in_first_or_second_pos(col, 'G')]

complete_df['Codons_with_g_at_first_or_second_pos'] = complete_df[columns_with_g_at_first_or_second_pos].sum(axis=1)
complete_df['Codons_without_g_at_first_or_second_pos'] = complete_df[columns_without_g_at_first_or_second_pos].sum(axis=1)

# **Running a statistical test N multiples times and plotting a distribution for P values, each one of them made with different 17-randomly-chosen variables.**



In [23]:
# we remove all the non-numerical columns
group_0 = complete_df[complete_df['Prion Formation'] == 0].iloc[:, 5:].drop('Protein Sequence', axis=1)
group_1 = complete_df[complete_df['Prion Formation'] == 1].iloc[:, 5:].drop('Protein Sequence', axis=1)

# Using Mann-Whitney U test for comparing the features between the two groups.

In [29]:
save_dir = '/content/drive/MyDrive/Lin Chen Rotation/Statistical Comparisons/Without_VTS1'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

results_dict = {col: {'p_values': [], 'higher_group': []} for col in group_0.columns}

for _ in tqdm.tqdm(range(10000)): # number of iterations (comparisons)
    random_group_0 = random_sample(group_0, 17)

    for col in group_0.columns:
      stat, p = mannwhitneyu(random_group_0[col], group_1[col], alternative='two-sided')
      results_dict[col]['p_values'].append(p)

      # Determine which group has higher values on average (median) in this iteration
      if random_group_0[col].median() > group_1[col].median():
        results_dict[col]['higher_group'].append('Non-prion-forming genes')
      else:
        results_dict[col]['higher_group'].append('Prion-forming genes')

for col, res in results_dict.items():
    p_values = np.array(res['p_values'])
    higher_group = np.array(res['higher_group'])
    num_significant = np.sum(p_values < 0.05)

    plt.figure(figsize=(12, 8))

    p_values_g0 = p_values[higher_group == 'Non-prion-forming genes']
    p_values_g1 = p_values[higher_group == 'Prion-forming genes']

    plt.hist(p_values_g0, bins=30, alpha=0.7, color='blue', label='Non-prion-forming genes > Prion-forming genes', edgecolor='black')
    plt.hist(p_values_g1, bins=30, alpha=0.7, color='red', label='Prion-forming genes > Non-prion-forming genes', edgecolor='black')

    plt.axvline(0.05, color='black', linestyle='--', label='p = 0.05')


    plt.title(f'P-Value Distribution for {col}')
    plt.xlabel('P-Value')
    plt.ylabel('Frequency')
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    y_max = plt.gca().get_ylim()[1]

    plt.text(0.1, y_max * 0.95, f'Significant p-values: {num_significant}',
         fontsize=12, color='black', bbox=dict(facecolor='white', alpha=0.6, edgecolor='black'))


    plt.savefig(os.path.join(save_dir, f'{col}_p_val_distribution.png'))

    plt.clf()
    # plt.show()

100%|██████████| 10000/10000 [17:26<00:00,  9.56it/s]
/tmp/ipykernel_6040/2947769364.py:25: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(12, 8))


<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

...

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

# **Preparing mutant libraries for WALTZ analysis.**

# Generating Mutant Sequences

In [ ]:
# for mutating amino acids
def generate_mutated_sequences(df, from_letter, to_letters, save_dir, split_pos):

    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    mutated_dfs = {}

    for _, row in df.iterrows():
        gene_name = row['Gene Name']
        prd_seq = row['PrD Protein Sequence']

        # find all positions of the source amino acid (to be mutated from)
        positions = [i for i, aa in enumerate(prd_seq) if aa == from_letter]

        if not positions:  # if letter isn't found, skip
            continue

        prd_start, _ = map(int, split_pos[gene_name].split('-'))

        # initialize list with the original sequence (first line in FASTA file)
        mutated_sequences = [{
            "ID": row["ID"],
            "Gene Name": gene_name,
            "PrD Protein Sequence": prd_seq
        }]

        # generate mutations and group them by position (such that SUP35;P41L is followed by SUP35;P41S, and so on..)
        for pos in positions:
            for to_letter in to_letters:
                new_prd_seq = list(prd_seq) # because lists are mutable, not strings
                new_prd_seq[pos] = to_letter
                original_position = pos + prd_start
                mutation_label = f"{from_letter}{original_position}{to_letter}"

                mutated_sequences.append({
                    "ID": row["ID"],
                    "Gene Name": f"{gene_name};{mutation_label}",
                    "PrD Protein Sequence": "".join(new_prd_seq), # convering list back to string
                })

        mutated_df = pd.DataFrame(mutated_sequences)

        fasta_path = os.path.join(save_dir, f"{gene_name}.fasta")
        with open(fasta_path, 'w') as fasta_file:
            for _, row in mutated_df.iterrows():
                fasta_file.write(f">{row['Gene Name']}\n{row['PrD Protein Sequence']}\n")

        mutated_dfs[gene_name] = mutated_df

    return mutated_dfs

In [ ]:
# for mutating nucleotides

codon_table = CodonTable.unambiguous_dna_by_id[1]
stop_codons = {"TAA", "TAG", "TGA"}

def mutate_and_translate_dna(dna_seq, mutate_pos, from_nt, to_nt):

    mutated_dna = list(dna_seq) # because lists are mutable, not strings
    mutated_dna[mutate_pos] = to_nt
    mutated_dna_seq = "".join(mutated_dna) # convering list back to string

    mutated_protein = str(Seq(mutated_dna_seq).translate(to_stop=False, table=codon_table)) # translating the mutated sequence

    for i, aa in enumerate(mutated_protein):
        if aa == "*":  # if a stop codon is found
            mutated_protein = mutated_protein[:i]  # truncate just before the stop codon..
            break

    return mutated_protein

def generate_mutated_sequences(df, from_nuc, to_nuc, save_dir, split_pos):

    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    mutated_dfs = {}

    for _, row in df.iterrows():
        gene_name = row['Gene Name']
        prd_dna_seq = row['Sequence']
        prd_prot_seq = row['Protein Sequence']

        # find all positions of the source amino acid (to be mutated from)
        positions = [i for i, nt in enumerate(prd_dna_seq) if nt == from_nuc]

        if not positions:  # if letter isn't found, skip
            continue

        prd_start, _ = map(int, split_pos[gene_name].split('-'))

        # Initialize list with the original sequence (first line in FASTA file)
        mutated_sequences = [{
            "ID": row["ID"],
            "Gene Name": gene_name,
            "Protein Sequence": prd_prot_seq
        }]

        for pos in positions: # for every position the nt to be mutated is found
            mutated_protein = mutate_and_translate_dna(prd_dna_seq, pos, from_nuc, to_nuc) # applying the mutate_and_translate function coded above to mutate the DNA sequence at that particular position

            # Identify affected amino acid
            aa_index = pos // 3  # Convert nucleotide index to amino acid index. If a nucleotide is at index 7 (pos 8), that means its floor division is index 2 (aa of pos 3).
            if aa_index >= len(prd_prot_seq):  # Safety check just in case truncations and stop codons mess with anything
                continue

            original_aa = prd_prot_seq[aa_index] # fetch the original amino acid in the protein
            mutated_aa = mutated_protein[aa_index] if aa_index < len(mutated_protein) else "*" # the mutated amino acid is at the same position as that found in the original protein (codon index), but using the mutated_protein string instead of prd_protein_seq
            # if the aa_index for some reason is larger than the length of the mutated protein, it means a truncation happened. In that case, the mutated amino acid is just "*"

            if original_aa == mutated_aa: # in case it's a synonmous mutation, don't add it to the FASTA file or the mutated_dfs dictionary.
                continue

            aa_position = aa_index + prd_start  # Adjust based on PrD start so that the position is relative to the full-length protein
            mutation_label = f"{gene_name};{original_aa}{aa_position}{mutated_aa}" # record the mutated sequence name for FASTA file and WALTZ later

            mutated_sequences.append({
                "ID": row["ID"],
                "Gene Name": mutation_label,
                "Protein Sequence": mutated_protein,
            })

        mutated_df = pd.DataFrame(mutated_sequences)

        fasta_path = os.path.join(save_dir, f"{gene_name}.fasta")
        with open(fasta_path, 'w') as fasta_file:
            for _, row in mutated_df.iterrows():
                fasta_file.write(f">{row['Gene Name']}\n{row['Protein Sequence']}\n")

        mutated_dfs[gene_name] = mutated_df

    return mutated_dfs

In [ ]:
save_dir = 'output/path'
mutated_dfs = generate_mutated_sequences(group_1, "C", "U", save_dir, split_positions)

In [ ]:
mutated_dfs['SUP35']

,ID,Gene Name,Protein Sequence
0,YDR172W,SUP35,MSDSNQGNNQQNYQQYSQNGNQQQGNNRYQGYQAYNAQAQPAGGYY...
1,YDR172W,SUP35;M1I,ISDSNQGNNQQNYQQYSQNGNQQQGNNRYQGYQAYNAQAQPAGGYY...
2,YDR172W,SUP35;D3N,MSNSNQGNNQQNYQQYSQNGNQQQGNNRYQGYQAYNAQAQPAGGYY...
3,YDR172W,SUP35;G7S,MSDSNQSNNQQNYQQYSQNGNQQQGNNRYQGYQAYNAQAQPAGGYY...
4,YDR172W,SUP35;G7D,MSDSNQDNNQQNYQQYSQNGNQQQGNNRYQGYQAYNAQAQPAGGYY...
5,YDR172W,SUP35;S17N,MSDSNQGNNQQNYQQYNQNGNQQQGNNRYQGYQAYNAQAQPAGGYY...
6,YDR172W,SUP35;G20S,MSDSNQGNNQQNYQQYSQNSNQQQGNNRYQGYQAYNAQAQPAGGYY...
7,YDR172W,SUP35;G20D,MSDSNQGNNQQNYQQYSQNDNQQQGNNRYQGYQAYNAQAQPAGGYY...
8,YDR172W,SUP35;G25S,MSDSNQGNNQQNYQQYSQNGNQQQSNNRYQGYQAYNAQAQPAGGYY...
9,YDR172W,SUP35;G25D,MSDSNQGNNQQNYQQYSQNGNQQQDNNRYQGYQAYNAQAQPAGGYY...
